<a href="https://colab.research.google.com/github/18217265596/sx/blob/master/LigandMPNN_Colab_Complete_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LigandMPNN → per-FASTA extract.py → ColabFold

支持多个 checkpoint 与多个 temperature 的笛卡尔积运行。每个输出 FASTA 独立提取 Top N，合并并按完整复合物序列去重后进入 ColabFold。

In [ ]:
# 0. 上传 PDB，并显示蛋白链顺序
from google.colab import files
from pathlib import Path
from collections import OrderedDict
import csv, hashlib, importlib.util, json, math, os, re, shutil, subprocess, sys

uploaded = files.upload()
items = [(n, b) for n, b in uploaded.items() if n.lower().endswith(".pdb")]
if len(items) != 1:
    raise ValueError("请一次只上传一个 .pdb 文件。")

ROOT = Path("/content/LigandMPNN")
INPUT_DIR = Path("/content/user_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
USER_PDB = INPUT_DIR / Path(items[0][0]).name
USER_PDB.write_bytes(items[0][1])

AA3 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
    "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
    "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M",
}
residues = OrderedDict()
for line in USER_PDB.read_text(errors="replace").splitlines():
    if line[:6].strip() not in {"ATOM","HETATM"} or line[12:16].strip() != "CA":
        continue
    if line[16:17] not in {" ","A"}:
        continue
    resname = line[17:20].strip().upper()
    if resname not in AA3:
        continue
    chain = line[21:22].strip() or "_"
    key = (line[22:26].strip(), line[26:27].strip())
    residues.setdefault(chain, OrderedDict()).setdefault(key, AA3[resname])

PDB_CHAIN_SEQUENCES = OrderedDict(
    (chain, "".join(seq.values())) for chain, seq in residues.items() if seq
)
PDB_CHAIN_ORDER = list(PDB_CHAIN_SEQUENCES)
if not PDB_CHAIN_ORDER:
    raise ValueError("PDB 中没有识别到标准蛋白链 CA 原子。")

print("PDB:", USER_PDB)
for i, chain in enumerate(PDB_CHAIN_ORDER, 1):
    print(f"{i}. chain {chain}: {len(PDB_CHAIN_SEQUENCES[chain])} aa")


In [ ]:
# 1. 所有用户参数（de novo 链在下一个独立 cell）
CHECKPOINT_OPTIONS = {
 1:("protein_mpnn","proteinmpnn_v_48_002.pt","ProteinMPNN 0.02 Å"),
 2:("protein_mpnn","proteinmpnn_v_48_010.pt","ProteinMPNN 0.10 Å"),
 3:("protein_mpnn","proteinmpnn_v_48_020.pt","ProteinMPNN 0.20 Å"),
 4:("protein_mpnn","proteinmpnn_v_48_030.pt","ProteinMPNN 0.30 Å"),
 5:("ligand_mpnn","ligandmpnn_v_32_005_25.pt","LigandMPNN 0.05 Å"),
 6:("ligand_mpnn","ligandmpnn_v_32_010_25.pt","LigandMPNN 0.10 Å"),
 7:("ligand_mpnn","ligandmpnn_v_32_020_25.pt","LigandMPNN 0.20 Å"),
 8:("ligand_mpnn","ligandmpnn_v_32_030_25.pt","LigandMPNN 0.30 Å"),
 9:("per_residue_label_membrane_mpnn","per_residue_label_membrane_mpnn_v_48_020.pt","MembraneMPNN per-residue"),
 10:("global_label_membrane_mpnn","global_label_membrane_mpnn_v_48_020.pt","MembraneMPNN global"),
 11:("soluble_mpnn","solublempnn_v_48_002.pt","SolubleMPNN 0.02 Å"),
 12:("soluble_mpnn","solublempnn_v_48_010.pt","SolubleMPNN 0.10 Å"),
 13:("soluble_mpnn","solublempnn_v_48_020.pt","SolubleMPNN 0.20 Å"),
 14:("soluble_mpnn","solublempnn_v_48_030.pt","SolubleMPNN 0.30 Å"),
 15:("sidechain_packer","ligandmpnn_sc_v_32_002_16.pt","仅用于侧链打包"),
}
for i,(_,fn,desc) in CHECKPOINT_OPTIONS.items(): print(f"{i:>2}: {fn:<48} | {desc}")

# LigandMPNN：支持单值 "6" 或多值 "1,2,3"；15 不能作为主任务。
TASK_CHECKPOINT_IDS = "1,2,3"
# temperature：支持单值 "0.1" 或多值 "0.1,0.01"。
TEMPERATURES = "0.1,0.01"
CHAINS_TO_DESIGN = "A"
SEED = 112
SEQUENCES_PER_COMBINATION = 100  # 每个 checkpoint × temperature 组合
BATCH_SIZE = 10                  # 必须整除上面的数量
PARSE_ATOMS_WITH_ZERO_OCCUPANCY = 1
SAVE_STATS = 1
FIXED_RESIDUES = ""
REDESIGNED_RESIDUES = ""
VERBOSE = 1
PACK_SIDE_CHAINS = False
NUMBER_OF_PACKS_PER_DESIGN = 1

# extract.py：每个输出 FASTA 单独取前 N 条，再合并、全局去重。
EXTRACT_SOURCE_GLOB = "*.fa"
EXTRACT_TOP_N_PER_FASTA = 20
EXTRACT_PER_FASTA_DIR_NAME = "extract_per_fasta"
EXTRACT_MERGED_FASTA_NAME = "merged_top_unique_sequences.fa"
EXTRACT_TSV_NAME = "merged_top_unique_sequences.tsv"

# ColabFold
RUN_COLABFOLD = True
COLABFOLD_MSA_SERVER = "https://api.colabfold.com"
COLABFOLD_USE_ENV = True
COLABFOLD_USE_FILTER = True
COLABFOLD_MODEL_TYPE = "alphafold2_multimer_v3"
COLABFOLD_NUM_RECYCLES = 3
COLABFOLD_NUM_MODELS = 3
COLABFOLD_MODEL_ORDER = [1,2,3]
COLABFOLD_NUM_SEEDS = 1
COLABFOLD_USE_DROPOUT = False
COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE = "auto"
COLABFOLD_MAX_MSA = "auto"
COLABFOLD_NUM_RELAX = 0
COLABFOLD_CALC_EXTRA_PTM = True
COLABFOLD_MAX_BINDERS = None  # 合并并去重后再限制总数
COLABFOLD_JOB_PREFIX = "binder_complex"
COLABFOLD_FINAL_CSV_NAME = "colabfold_ranked_iptm_ptm.csv"
COLABFOLD_DELETE_INTERMEDIATES = True
DOWNLOAD_FINAL_CSV = True

def parse_multi(value,cast,name):
    if isinstance(value,(list,tuple,set)): raw=list(value)
    elif isinstance(value,(int,float)): raw=[value]
    else: raw=[x.strip() for x in str(value).replace("，",",").split(",") if x.strip()]
    if not raw: raise ValueError(f"{name} 不能为空。")
    try: out=[cast(x) for x in raw]
    except (TypeError,ValueError) as e: raise ValueError(f"{name} 含无法解析的值：{raw}") from e
    return list(dict.fromkeys(out))

TASK_CHECKPOINT_ID_LIST=parse_multi(TASK_CHECKPOINT_IDS,int,"TASK_CHECKPOINT_IDS")
TEMPERATURE_LIST=parse_multi(TEMPERATURES,float,"TEMPERATURES")
bad=[x for x in TASK_CHECKPOINT_ID_LIST if x not in range(1,15)]
if bad: raise ValueError(f"主任务 checkpoint 只能为 1–14：{bad}")
if any(not math.isfinite(x) or x<=0 for x in TEMPERATURE_LIST): raise ValueError("temperature 必须为正的有限数值。")
if not isinstance(SEQUENCES_PER_COMBINATION,int) or isinstance(SEQUENCES_PER_COMBINATION,bool) or SEQUENCES_PER_COMBINATION<1: raise ValueError("SEQUENCES_PER_COMBINATION 必须为正整数。")
if not isinstance(BATCH_SIZE,int) or isinstance(BATCH_SIZE,bool) or BATCH_SIZE<1: raise ValueError("BATCH_SIZE 必须为正整数。")
if SEQUENCES_PER_COMBINATION%BATCH_SIZE: raise ValueError("SEQUENCES_PER_COMBINATION 必须能被 BATCH_SIZE 整除。")
NUMBER_OF_BATCHES=SEQUENCES_PER_COMBINATION//BATCH_SIZE
if FIXED_RESIDUES.strip() and REDESIGNED_RESIDUES.strip(): raise ValueError("FIXED_RESIDUES 与 REDESIGNED_RESIDUES 不能同时使用。")
if not isinstance(EXTRACT_TOP_N_PER_FASTA,int) or EXTRACT_TOP_N_PER_FASTA<1: raise ValueError("EXTRACT_TOP_N_PER_FASTA 必须为正整数。")
if COLABFOLD_MAX_BINDERS is not None and int(COLABFOLD_MAX_BINDERS)<1: raise ValueError("COLABFOLD_MAX_BINDERS 必须为 None 或正整数。")
if COLABFOLD_MODEL_TYPE!="alphafold2_multimer_v3": raise ValueError("模型固定为 alphafold2_multimer_v3。")
if COLABFOLD_NUM_MODELS!=3 or COLABFOLD_MODEL_ORDER!=[1,2,3]: raise ValueError("固定使用模型 1、2、3。")
if COLABFOLD_NUM_RELAX!=0: raise ValueError("当前流程不做 Amber relaxation。")
if not COLABFOLD_CALC_EXTRA_PTM: raise ValueError("请保持 COLABFOLD_CALC_EXTRA_PTM=True。")
if isinstance(COLABFOLD_NUM_RECYCLES,str) and COLABFOLD_NUM_RECYCLES!="auto": raise ValueError("recycles 只能为整数或 auto。")
RUN_COMBINATIONS=[(c,t) for c in TASK_CHECKPOINT_ID_LIST for t in TEMPERATURE_LIST]

print('\n输入示例：TASK_CHECKPOINT_IDS="1,2,3"；TEMPERATURES="0.1,0.01"')
print("checkpoint:",TASK_CHECKPOINT_ID_LIST)
print("temperature:",TEMPERATURE_LIST)
print("组合数:",len(RUN_COMBINATIONS))
print("每个组合生成:",SEQUENCES_PER_COMBINATION)
print("理论总数:",len(RUN_COMBINATIONS)*SEQUENCES_PER_COMBINATION)
print("每个 FASTA 提取:",EXTRACT_TOP_N_PER_FASTA)


In [ ]:
# 2. 指定 de novo 链；留空表示所有链都做 MSA
COLABFOLD_DE_NOVO_CHAIN = "A"

COLABFOLD_DE_NOVO_CHAIN = COLABFOLD_DE_NOVO_CHAIN.strip()
if "," in COLABFOLD_DE_NOVO_CHAIN:
    raise ValueError("当前只支持一条 de novo 链。")
if COLABFOLD_DE_NOVO_CHAIN and COLABFOLD_DE_NOVO_CHAIN not in PDB_CHAIN_ORDER:
    raise ValueError(f"可用链：{PDB_CHAIN_ORDER}")
print("de novo chain:", COLABFOLD_DE_NOVO_CHAIN or "None（所有链均使用 MSA）")


In [ ]:
# 3. 下载并执行同一 GitHub 仓库中的生产流程脚本
PIPELINE_FILES = [
    (
        "ligandmpnn_matrix_pipeline.py",
        "https://raw.githubusercontent.com/18217265596/sx/master/"
        "ligandmpnn_matrix_pipeline.py",
    ),
    (
        "colabfold_scoring_pipeline.py",
        "https://raw.githubusercontent.com/18217265596/sx/master/"
        "colabfold_scoring_pipeline.py",
    ),
]

for filename, url in PIPELINE_FILES:
    local_path = Path("/content") / filename
    subprocess.run(
        ["wget", "-q", url, "-O", str(local_path)],
        check=True,
    )
    if not local_path.exists() or local_path.stat().st_size == 0:
        raise RuntimeError(f"流程脚本下载失败：{filename}")
    print("Running:", local_path)
    exec(
        compile(
            local_path.read_text(encoding="utf-8"),
            str(local_path),
            "exec",
        ),
        globals(),
    )


## 多 checkpoint / 多 temperature 规则

- `TASK_CHECKPOINT_IDS` 可填 `"6"` 或 `"1,2,3"`；`TEMPERATURES` 可填 `"0.1"` 或 `"0.1,0.01"`。
- 每个 checkpoint × temperature 组合精确生成 `SEQUENCES_PER_COMBINATION` 条；该数必须能被 `BATCH_SIZE` 整除。
- `extract.py` 对每个输出 FASTA 单独提取 `EXTRACT_TOP_N_PER_FASTA` 条，再合并，并按完整复合物序列全局去重。
- 重复序列只送 ColabFold 一次；合并清单保留 `overall_confidence` 最高的来源 checkpoint、temperature 和 FASTA。
- `COLABFOLD_MAX_BINDERS` 在合并和全局去重后生效。
- 指定 de novo 链时最终 CSV 为 `de_novo_sequence,total_sequence,iptm,ptm`；留空时为 `total_sequence,iptm,ptm`。
- 每条候选从模型 1–3 中选 ipTM 最高者；没有 ipTM 时回退 pTM。最终 CSV 同样按此规则降序。